# TUM Dataset

In [ ]:
%load_ext autoreload
%autoreload 2
%xmode Context

import sys
sys.tracebacklimit = None

print(__debug__)
from headset_data import *
from robot_environment import *

In [ ]:
from load_from_tum import *

tum_rgbd_dataset_location = "/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/datasets/rgbd_dataset_freiburg2_desk"

tum_robot_env, tum_headset_data = robot_environment_and_headset_data_from_tum(
        folder=tum_rgbd_dataset_location,
        rgb_camera_name="freiburg2",
        time_tolerance= 0.03,
        n_robot_images= 20,
        xyz_image_generation_config=XYZImageGenerationConfig(),
        xyz_image_alginment_config=ICPAlignmentConfig(do_alginment=False),
        intervall=(0.1, 0.2)
)

tum_robot_env.save("tum")
tum_headset_data.save("tum")
tum_robot_env = RobotEnvironment.from_folder("/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/pose_estimation/tum/rgbd_dataset_freiburg2_desk_robot_env")
tum_headset_data = HeadsetData.from_folder("/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/pose_estimation/tum/rgbd_dataset_freiburg2_desk_headset_data")

In [ ]:
#from pose_pred_points_ellipsoids import *
#
#s3_seg1 = SAM3Segmenter(Sam3Prompt())
#s3_seg2 = SAM3Segmenter(Sam3Prompt())
#image = tum_headset_data.bgr_image_s[0]
#for i, image in enumerate(tum_robot_env.robot_bgr_images):
#    print(f"frame: {i} num detect: {s3_seg1.get_object_masks(image, visualize=False).shape[0]}")    
#for i, image in enumerate(tum_headset_data.bgr_image_s):
#    print(f"frame: {i} num detect: {s3_seg2.get_object_masks(image, visualize=False).shape[0]}")    

### 3d visualisation

In [ ]:
vis_robot_env, vis_headset, vis_both = False, False, True

if vis_robot_env:
    tum_robot_env.visualize_3d_data()
if vis_headset:
    tum_headset_data.visualize_3d_data()
if vis_both:
    visualize_robot_camera_environment_combo(robot_env=tum_robot_env, headset_data=tum_headset_data)

## Testing Predictors

In [ ]:
from predictor_grader import *
from pose_pred_points import *
from pose_pred_points_lines import *
from pose_pred_points_ellipsoids import *

In [ ]:
# Creation of the Predictors
light_glue = ExtractAndLightGlue()

no_ellips = GradablePosePredictor(
    creator=OnlyPointsPredictor.get_creation_function(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Augmentation],
            extract_and_match=light_glue,
            ransac_config=pose_estimation_ransaac_config_precise,
        )
    ),
    name="no ellipse"
)

ellips_adam = GradablePosePredictor(
    creator=EllipsoidPredictor.get_creation_function(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Augmentation],
            extract_and_match=light_glue,
            ransac_config=pose_estimation_ransaac_config_less_precise,
        ),
        pne_optimizer=PnEDeltaPoseAdamOptimizer(PnEDeltaPoseAdamOptimizerConfig(learning_rate=0.005, max_itterations = 1000)),
        ellipsoid_refinement_at_res=(1400, 1400),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
        cam2_segmenter=YOLOv26Segmenter("yoloe-26l-seg.pt"),
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        visualize_pne_optimisation=False,
        visualize_ellipsoid_fitting=True,
        visualize_environment_generation = True,
        visualize_segmentation_masks=False
    ),
    name="ellips_adam"
)

In [ ]:
grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=[ellips_adam],
    headset_data = tum_headset_data,
    robot_env = tum_robot_env,
)

In [ ]:
visualize_trajectories_3d = False
if visualize_trajectories_3d:
    grader.visualize_predictions_3d()

grader.print_summary()

fig1, ax1 = plt.subplots(1, 1, figsize = (16, 8))
grader.plot_translational_errors(ax1)

fig2, ax2 = plt.subplots(1, 1, figsize = (16, 8))
grader.plot_rotational_errors(ax2)

fig3, ax3 = plt.subplots(1, 1, figsize = (16, 8))
grader.plot_creation_times(ax3)

fig4, ax4 = plt.subplots(1, 1, figsize = (8, 5))
grader.plot_successful_frame_prediction_times(ax4)


plt.show()

In [ ]:
from geometric_utilities.slam2mp4 import VideoGenerator

video_predictor = EllipsoidPredictor(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=tum_robot_env.robot_bgr_images,
        cam1_xyz_images=tum_robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Augmentation],
            extract_and_match=light_glue,
            ransac_config=pose_estimation_ransaac_config_less_precise,
        ),
        pne_optimizer=PyposePNEOptimizer(),
        ellipsoid_refinement_at_res=(1400, 1400),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt(text="Items on a desk")),
        cam2_segmenter=yolo,
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        visualize_pne_optimisation=False
)

init_predictor_grade = PredictionOnDataset(
    predictor = video_predictor,
    headset_data = tum_headset_data,
    number_retry = 1,
    vid_gen=VideoGenerator(fps=20),
    video_save_location="tum.mp4",
    point_cloud=tum_robot_env.robot_xyz_images.reshape(-1,3)
)
init_predictor_grade.print_summary()